In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score
import kagglehub

# 1. Выбор начальных условий

## Выбор датасета
Был выбран набор данных Intel Image Classification, содержащий около 17 тысяч размеченных изображений природных и городских пейзажей, распределенных по 6 категориям:

Buildings (Здания)

Forest (Лес)

Glacier (Ледник)

Mountain (Горы)

Sea (Море)

Street (Улица)

Практическая значимость: Автоматическое распознавание сцен может применяться в геоинформационных системах, для навигации беспилотных летательных аппаратов (дронов), в системах автопилота для анализа окружающей среды, а также для автоматической тегировки и сортировки пользовательских фотографий.

## Метрики качества
Для оценки качества моделей выбраны:

Accuracy (Точность) — базовая метрика, показывающая общую долю правильных ответов.

F1-Score (Макро-усреднение) — гармоническое среднее между Precision и Recall. Так как некоторые классы могут иметь схожие визуальные признаки (например, горы и ледники), F1-score позволит объективно оценить качество классификации каждого класса в отдельности.

# 2. Создание бейзлайна и оценка качества
## Загрузка датасета

In [ ]:
dataset_path = kagglehub.dataset_download("puneet6060/intel-image-classification")

TRAIN_DIR = os.path.join(dataset_path, "seg_train", "seg_train")
VAL_DIR = os.path.join(dataset_path, "seg_test", "seg_test")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=base_transform)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=base_transform)

num_classes = len(train_dataset.classes)
print(f"Найдено классов: {num_classes} ({train_dataset.classes})")
print(f"Размер тренировочной выборки: {len(train_dataset)}")
print(f"Размер тестовой выборки: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

Using Colab cache for faster access to the 'intel-image-classification' dataset.
Используемое устройство: cuda
Найдено классов: 6 (['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street'])
Размер тренировочной выборки: 14034
Размер тестовой выборки: 3000


## Функция обучения

In [ ]:
def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")

    return acc, f1

criterion = nn.CrossEntropyLoss()

## Обучение бейзлайнов (ResNet18 и ViT_B_16)

In [ ]:
print("--- Запуск бейзлайна: ResNet18 ---")
resnet_base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = resnet_base.fc.in_features
resnet_base.fc = nn.Linear(num_ftrs, num_classes)
resnet_base = resnet_base.to(device)

optimizer_resnet = optim.Adam(resnet_base.parameters(), lr=0.001)
acc_res_base, f1_res_base = train_and_evaluate(resnet_base, train_loader, val_loader, criterion, optimizer_resnet, epochs=5)

print("\n--- Запуск бейзлайна: Vision Transformer (ViT_B_16) ---")
vit_base = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
for param in vit_base.parameters():
    param.requires_grad = False =

num_ftrs_vit = vit_base.heads.head.in_features
vit_base.heads.head = nn.Linear(num_ftrs_vit, num_classes)
vit_base = vit_base.to(device)

optimizer_vit = optim.Adam(vit_base.parameters(), lr=1e-5)
acc_vit_base, f1_vit_base = train_and_evaluate(vit_base, train_loader, val_loader, criterion, optimizer_vit, epochs=5)

--- Запуск бейзлайна: ResNet18 ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 235MB/s]


Epoch 1/5 | Train Loss: 0.4781 | Val Acc: 0.8797 | Val F1: 0.8805
Epoch 2/5 | Train Loss: 0.3364 | Val Acc: 0.8803 | Val F1: 0.8812
Epoch 3/5 | Train Loss: 0.2774 | Val Acc: 0.8513 | Val F1: 0.8534
Epoch 4/5 | Train Loss: 0.2429 | Val Acc: 0.8853 | Val F1: 0.8859
Epoch 5/5 | Train Loss: 0.2117 | Val Acc: 0.8890 | Val F1: 0.8891

--- Запуск бейзлайна: Vision Transformer (ViT_B_16) ---
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:04<00:00, 76.7MB/s]


Epoch 1/5 | Train Loss: 1.5457 | Val Acc: 0.6157 | Val F1: 0.6144
Epoch 2/5 | Train Loss: 1.1493 | Val Acc: 0.7883 | Val F1: 0.7901
Epoch 3/5 | Train Loss: 0.8952 | Val Acc: 0.8367 | Val F1: 0.8378
Epoch 4/5 | Train Loss: 0.7311 | Val Acc: 0.8580 | Val F1: 0.8598
Epoch 5/5 | Train Loss: 0.6209 | Val Acc: 0.8770 | Val F1: 0.8787


# 3. Улучшение бейзлайна
## Формулировка гипотез

Гипотеза 1: Применение аугментации данных (случайные отражения, небольшие повороты) поможет модели не переобучаться на специфичных пейзажах тренировочной выборки и лучше выделять ключевые объекты (деревья, здания).

Гипотеза 2: Использование планировщика скорости обучения (StepLR) позволит алгоритму точнее сойтись в минимум функции потерь.

## Код улучшенного бейзлайна:

In [ ]:
augmented_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset_aug = datasets.ImageFolder(root=TRAIN_DIR, transform=augmented_transform)
val_dataset_aug = datasets.ImageFolder(root=VAL_DIR, transform=base_transform)

train_loader_aug = DataLoader(train_dataset_aug, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader_aug = DataLoader(val_dataset_aug, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

def train_and_evaluate_with_scheduler(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        scheduler.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')
        print(f"Epoch {epoch+1}/{epochs} | LR: {scheduler.get_last_lr()[0]:.6f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")
    return acc, f1

print("--- Запуск улучшенного бейзлайна: ResNet18 ---")
resnet_improved = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet_improved.fc = nn.Linear(resnet_improved.fc.in_features, num_classes)
resnet_improved = resnet_improved.to(device)

optimizer_imp = optim.Adam(resnet_improved.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer_imp, step_size=3, gamma=0.1)

acc_res_imp, f1_res_imp = train_and_evaluate_with_scheduler(
    resnet_improved, train_loader_aug, val_loader_aug, criterion, optimizer_imp, scheduler, epochs=5
)

print("\n--- Запуск улучшенного бейзлайна: ViT_B_16 ---")
vit_improved = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
for param in vit_improved.parameters():
    param.requires_grad = False
vit_improved.heads.head = nn.Linear(vit_improved.heads.head.in_features, num_classes)
vit_improved = vit_improved.to(device)

optimizer_vit_imp = optim.Adam(vit_improved.heads.head.parameters(), lr=0.001)
scheduler_vit = optim.lr_scheduler.StepLR(optimizer_vit_imp, step_size=3, gamma=0.1)

acc_vit_imp, f1_vit_imp = train_and_evaluate_with_scheduler(
    vit_improved, train_loader_aug, val_loader_aug, criterion, optimizer_vit_imp, scheduler_vit, epochs=5
)

--- Запуск улучшенного бейзлайна: ResNet18 ---
Epoch 1/5 | LR: 0.001000 | Val Acc: 0.8060 | Val F1: 0.8009
Epoch 2/5 | LR: 0.001000 | Val Acc: 0.9030 | Val F1: 0.9042
Epoch 3/5 | LR: 0.000100 | Val Acc: 0.9077 | Val F1: 0.9099
Epoch 4/5 | LR: 0.000100 | Val Acc: 0.9223 | Val F1: 0.9237
Epoch 5/5 | LR: 0.000100 | Val Acc: 0.9217 | Val F1: 0.9226

--- Запуск улучшенного бейзлайна: ViT_B_16 ---
Epoch 1/5 | LR: 0.001000 | Val Acc: 0.9310 | Val F1: 0.9329
Epoch 2/5 | LR: 0.001000 | Val Acc: 0.9353 | Val F1: 0.9371
Epoch 3/5 | LR: 0.000100 | Val Acc: 0.9320 | Val F1: 0.9334
Epoch 4/5 | LR: 0.000100 | Val Acc: 0.9373 | Val F1: 0.9389
Epoch 5/5 | LR: 0.000100 | Val Acc: 0.9380 | Val F1: 0.9396


## Сравнение результатов
Сравнивая результаты ResNet18 Base и ResNet18 Improved, можно сделать вывод, что гипотеза подтвердилась.

| Метрика | ResNet18 | ResNet18 Improved |
| :--- | :--- | :--- |
| **Accuracy** | 0.8890 | 0.9217 |
| **F1-score** | 0.8891 | 0.9226 |

Аугментации помогают модели не переобучиться на тренировочной выборке. Снижение Learning Rate на 3-й эпохе позволило алгоритму точнее скорректировать веса (спуск в локальный минимум), что отразилось в уверенном росте метрик на 4-й и 5-й эпохах.

При сравнении ViT_B_16 и ViT_B_16 Improved можно сделать вывод, что гипотеза также полностью подтвердилась. Наблюдается колоссальный рост метрик.

| Метрика | ViT_B_16 | ViT_B_16 Improved |
| :--- | :--- | :--- |
| **Accuracy** | 0.8770 | 0.9380 |
| **F1-score** | 0.8787 | 0.9396 |

В отличие от работы со сложными и несбалансированными датасетами, на наборе природных пейзажей комбинация аугментаций и правильно подобранного стартового Learning Rate позволила трансформеру преодолеть проблему медленной сходимости. В результате улучшенная версия ViT не только побила свой бейзлайн, но и обогнала архитектуру ResNet18.

# 4. Имплементация алгоритма машинного обученияэ
В данном разделе мы отходим от использования готовых архитектур из библиотеки torchvision и самостоятельно реализуем две модели: сверточную (CustomCNN) и трансформерную (CustomViT). Это позволит оценить разницу между специализированными блоками и глубокими предобученными сетями.

## Самостоятельная имплементация моделей

### Реализация CustomCNN
Мы создаем архитектуру, состоящую из трех последовательных блоков. Каждый блок включает свертку (Conv2d) для извлечения признаков, функцию активации ReLU и слой подвыборки (MaxPool2d) для уменьшения размерности.

### Реализация CustomViT (Mini-ViT)
Модель реализует классический механизм Vision Transformer:

*   Patch Embedding: Изображение разбивается на патчи (фрагменты), которые проецируются в векторное пространство.

*   Positional Encoding: Добавление информации о положении патча в пространстве.

*   Transformer Encoder: Многослойная структура с механизмом Multi-Head Self-Attention (многоголовое самовнимание), позволяющая модели искать глобальные связи между частями изображения.

In [ ]:
import torch.nn.functional as F

# --- Custom CNN Implementation ---
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # После 3 пулингов размер 224x224 становится 28x28
        self.fc = nn.Linear(64 * 28 * 28, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# --- Custom Vision Transformer Implementation ---
class CustomViT(nn.Module):
    def __init__(self, num_classes, img_size=224, patch_size=32, dim=128, depth=3, heads=4, mlp_dim=256, dropout=0.1):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)

        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=mlp_dim,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.mlp_head = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.dropout(x + self.pos_embedding)
        x = self.transformer(x)
        return self.mlp_head(x[:, 0])

# --- Обучение кастомных моделей ---

print("--- Запуск CustomCNN (Базовая) ---")
custom_cnn = CustomCNN(num_classes).to(device)
optimizer_cnn = optim.Adam(custom_cnn.parameters(), lr=0.001)
acc_cnn_base, f1_cnn_base = train_and_evaluate(custom_cnn, train_loader, val_loader, criterion, optimizer_cnn, epochs=5)

print("\n--- Запуск CustomViT (Базовая) ---")
custom_vit = CustomViT(num_classes).to(device)
optimizer_vit_c = optim.Adam(custom_vit.parameters(), lr=0.0003)
acc_vit_c_base, f1_vit_c_base = train_and_evaluate(custom_vit, train_loader, val_loader, criterion, optimizer_vit_c, epochs=5)

# --- Обучение кастомных моделей с улучшениями (Аугментации + Scheduler) ---

print("\n--- Запуск CustomCNN (Улучшенная) ---")
custom_cnn_imp = CustomCNN(num_classes).to(device)
optimizer_cnn_imp = optim.Adam(custom_cnn_imp.parameters(), lr=0.001)
scheduler_cnn = optim.lr_scheduler.StepLR(optimizer_cnn_imp, step_size=3, gamma=0.1)
acc_cnn_imp, f1_cnn_imp = train_and_evaluate_with_scheduler(custom_cnn_imp, train_loader_aug, val_loader_aug, criterion, optimizer_cnn_imp, scheduler_cnn, epochs=5)

print("\n--- Запуск CustomViT (Улучшенная) ---")
custom_vit_imp = CustomViT(num_classes).to(device)
optimizer_vit_c_imp = optim.Adam(custom_vit_imp.parameters(), lr=0.0003)
scheduler_vit_c = optim.lr_scheduler.StepLR(optimizer_vit_c_imp, step_size=3, gamma=0.1)
acc_vit_c_imp, f1_vit_c_imp = train_and_evaluate_with_scheduler(custom_vit_imp, train_loader_aug, val_loader_aug, criterion, optimizer_vit_c_imp, scheduler_vit_c, epochs=5)

--- Запуск CustomCNN (Базовая) ---
Epoch 1/5 | Train Loss: 0.8784 | Val Acc: 0.7457 | Val F1: 0.7471
Epoch 2/5 | Train Loss: 0.5595 | Val Acc: 0.8163 | Val F1: 0.8184
Epoch 3/5 | Train Loss: 0.4025 | Val Acc: 0.7847 | Val F1: 0.7836
Epoch 4/5 | Train Loss: 0.2573 | Val Acc: 0.7937 | Val F1: 0.7905
Epoch 5/5 | Train Loss: 0.1483 | Val Acc: 0.7860 | Val F1: 0.7863

--- Запуск CustomViT (Базовая) ---
Epoch 1/5 | Train Loss: 1.1767 | Val Acc: 0.5973 | Val F1: 0.5914
Epoch 2/5 | Train Loss: 0.9740 | Val Acc: 0.6133 | Val F1: 0.6011
Epoch 3/5 | Train Loss: 0.9133 | Val Acc: 0.6497 | Val F1: 0.6455
Epoch 4/5 | Train Loss: 0.8815 | Val Acc: 0.6600 | Val F1: 0.6567
Epoch 5/5 | Train Loss: 0.8505 | Val Acc: 0.6610 | Val F1: 0.6519

--- Запуск CustomCNN (Улучшенная) ---
Epoch 1/5 | LR: 0.001000 | Val Acc: 0.7060 | Val F1: 0.7081
Epoch 2/5 | LR: 0.001000 | Val Acc: 0.7850 | Val F1: 0.7853
Epoch 3/5 | LR: 0.000100 | Val Acc: 0.8113 | Val F1: 0.8130
Epoch 4/5 | LR: 0.000100 | Val Acc: 0.8387 | Val F

## Сравнение результатов и выводы

### Сравнение с бейзлайном

Кастомные модели, обученные «с нуля», показывают результаты ниже, чем предобученные глубокие сети из библиотеки torchvision. Это наглядно демонстрирует эффективность метода Transfer Learning (переноса обучения) на предобученных весах ImageNet.

| Метрика | ResNet18 (Baseline) | CustomCNN (Base) |
| :--- | :--- | :--- |
| **Accuracy** | 0.8890 | 0.7860 |
| **F1-score** | 0.8891 | 0.7863 |

| Метрика | ViT_B_16 (Baseline) | CustomViT (Base) |
| :--- | :--- | :--- |
| **Accuracy** | 0.8770 | 0.6610 |
| **F1-score** | 0.8787 | 0.6519 |

### Сравнение улучшенных кастомных моделей

Добавление техник из улучшенного бейзлайна (аугментации данных и планировщик скорости обучения StepLR) оказало решающее влияние на сверточную сеть, но практически не изменило картину для кастомного трансформера.

| Метрика | CustomCNN (Base) | CustomCNN (Improved) |
| :--- | :--- | :--- |
| **Accuracy** | 0.7860 | 0.8283 |
| **F1-score** | 0.7863 | 0.8300 |

| Метрика | CustomViT (Base) | CustomViT (Improved) |
| :--- | :--- | :--- |
| **Accuracy** | 0.6610 | 0.6663 |
| **F1-score** | 0.6519 | 0.6624 |

## Итоговые выводы по заданию 4

* Особенности архитектур: Сверточные сети (CustomCNN) даже "с нуля" справляются с малым объемом данных лучше трансформеров, так как изначально заточены под поиск локальных паттернов (линий, текстур). Трансформерам (CustomViT) 14 000 изображений критически мало для выучивания глобальных связей с нуля (82.8% против 66.6%).

* Влияние улучшений: Внедрение аугментаций и StepLR успешно спасло CustomCNN от переобучения (рост точности на 4.2%). Однако для CustomViT искажения картинок стали лишь лишним шумом, не дав значимого прироста (<0.5%).

* Главный итог: Самописные модели ожидаемо уступают готовым решениям из torchvision. Для реальных инженерных задач классификации пейзажей необходимо применять Transfer Learning (предобученные сети), так как обучение с нуля требует несоразмерно больших объемов данных.

# Общий вывод по лабораторной работе №1
В ходе выполнения лабораторной работы был успешно реализован полный цикл исследования моделей компьютерного зрения (Computer Vision) на примере задачи многоклассовой классификации пейзажей (Intel Image Classification).

На основе полученных результатов можно сделать следующие ключевые выводы:

* Эффективность Transfer Learning: Использование предобученных на ImageNet моделей является абсолютным стандартом для решения прикладных задач. Базовые архитектуры (ResNet18 и ViT_B_16) показали точность около 88% практически "из коробки", в то время как самописные сети, обучаемые с нуля, с трудом преодолевают порог в 65-80%.

* Специфика архитектур: Сверточные сети (CNN) обладают сильным индуктивным смещением — они изначально хорошо ищут локальные паттерны, что делает их идеальными для небольших датасетов. Трансформеры (ViT), опирающиеся на механизм самовнимания (Self-Attention), обладают более высоким "потолком" точности (в нашей работе — 93.8%), но требуют ювелирной настройки гиперпараметров (Learning Rate) и огромных объемов данных для обучения с нуля.

* Роль регуляризации: Применение аугментации данных и планировщиков скорости обучения (StepLR) доказало свою эффективность. Эти методы успешно предотвращают переобучение моделей на специфичных выборках и помогают алгоритму оптимизации находить более глубокие локальные минимумы, повышая итоговую точность до 3-6%.

В результате был сформирован надежный ML-пайплайн, а лучшая модель (ViT_B_16 Improved) достигла метрики Accuracy 93.8%, что полностью удовлетворяет требованиям к разработке proof-of-concept решений для бизнеса.